# YAMNet Training

Transfer learning. YAMNet is frozen and only a small head is trained on its
embeddings, so this runs fast even without a GPU.

In [ ]:
# Colab already has TensorFlow, so we only pin the audio and sklearn parts.
!pip install -q scikit-learn==1.9.1 librosa==1.0.0 soundfile==0.14.0 joblib==1.6.0 tensorflow_hub==0.16.1

print()
print("If Colab shows a 'Restart runtime' button above, click it, then carry on")
print("from the Drive mount cell. Do not run this install cell again.")

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:

PROJECT_PATH = '/content/drive/MyDrive/SonicSentinel'

In [ ]:
import os
import sys

if not os.path.isdir(PROJECT_PATH):
    raise RuntimeError(f"PROJECT_PATH does not exist: {PROJECT_PATH}")

if not os.path.isfile(os.path.join(PROJECT_PATH, 'feature_extraction', 'embeddings.py')):
    raise RuntimeError(f"{PROJECT_PATH} does not look like the project folder.")

if PROJECT_PATH not in sys.path:
    sys.path.insert(0, PROJECT_PATH)

# Embeddings come from the project module, never redefined here.
try:
    import feature_extraction.embeddings as embeddings_module
    from feature_extraction.embeddings import EMBEDDING_LENGTH
    from config.config import CLASSES, PYTHON_MODELS
    from src import yamnet_training
except Exception as error:
    raise RuntimeError("Could not import the project modules from PROJECT_PATH.") from error

if not os.path.abspath(embeddings_module.__file__).startswith(os.path.abspath(PROJECT_PATH)):
    raise RuntimeError("embeddings.py was imported from the wrong place.")

print("imported the project modules from", PROJECT_PATH)
print("embeddings module:", embeddings_module.__file__)
print("embedding length:", EMBEDDING_LENGTH)
print("classes:", len(CLASSES))

In [ ]:
# Downloads YAMNet into python_models/yamnet so the app works offline later.
from feature_extraction.embeddings import download_yamnet, yamnet_is_downloaded

if yamnet_is_downloaded():
    print("YAMNet is already on Drive")
else:
    download_yamnet()

In [ ]:
MODEL_NAME = "yamnet"

# How many head settings to try. Each one trains in seconds.
CANDIDATES = 8

TUNING_EPOCHS = 30
FINAL_EPOCHS = 80

print("search space:", yamnet_training.SEARCH_SPACE)
print("combinations in total:", yamnet_training.count_space())

In [ ]:
# Builds embeddings, tunes on validation, retrains the best, scores test once.
# Embeddings are cached to data/metadata/*_embeddings.npz on Drive.
model = yamnet_training.run_training(
    candidates=CANDIDATES,
    tuning_epochs=TUNING_EPOCHS,
    final_epochs=FINAL_EPOCHS,
)

In [ ]:
# What this notebook wrote to Drive.
import os

names = [PYTHON_MODELS[MODEL_NAME]['model_file'], 'label_encoder.pkl']

for name in names:
    path = os.path.join(PROJECT_PATH, 'python_models', name)
    print('saved' if os.path.exists(path) else 'MISSING', 'python_models/' + name)

path = os.path.join(PROJECT_PATH, 'reports', PYTHON_MODELS[MODEL_NAME]['metrics_file'])
print('saved' if os.path.exists(path) else 'MISSING', 'reports/' + PYTHON_MODELS[MODEL_NAME]['metrics_file'])

print()
print("Download those into the same folders in the project. The YAMNet head")
print("needs no scaler. Set ACTIVE_PYTHON_MODEL = 'yamnet' in config/config.py")
print("and copy python_models/yamnet/ across too.")

In [ ]:
# The figures the training run wrote, shown here so they can be saved for the
# report. They are also on Drive in reports/plots/.
import glob
import os

from IPython.display import Image, display

pattern = os.path.join(PROJECT_PATH, "reports", "plots", "*.png")
paths = sorted(glob.glob(pattern))

mine = [p for p in paths if os.path.basename(p).startswith(MODEL_NAME)]
shared = [p for p in paths if os.path.basename(p) in
          ("model_comparison.png", "dataset_distribution.png", "example_spectrograms.png")]

if not paths:
    print("No plots found. Did the training cell finish?")

for path in mine + shared:
    print(os.path.basename(path))
    display(Image(filename=path))